# 05 XAI — Análise Explicável com LightGBM

**Repositório:** [FABRICIOBARILI/DOUTORADO](https://github.com/FABRICIOBARILI/DOUTORADO)
Dados: Google Cloud Storage | Projeto GCP: `doutorado-501917` | Bucket: `2025_rides`

> **DADOS MOVIDOS DO GOOGLE DRIVE PARA O GOOGLE CLOUD STORAGE.** Ganho em velocidade de leitura.

In [ ]:
# ── SINCRONIZAR COM GITHUB ──────────────────────────────────────────────────
# Execute esta célula para puxar a versão mais recente do repositório.
import os

REPO_URL = "https://github.com/FABRICIOBARILI/DOUTORADO.git"
REPO_DIR = "/content/DOUTORADO"
BRANCH   = "feat/changelog-inicial"   # ajuste conforme o branch ativo

if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"\n✅ Diretório atual: {os.getcwd()}")

In [ ]:
!pip install -q gcsfs duckdb
import pandas as pd
import gcsfs
from google.colab import auth
import duckdb

# 1. Garante a autenticação nativa do Colab
auth.authenticate_user()

# 2. Inicializa o FileSystem do GCS apontando para o seu projeto
project_id = 'doutorado-501917'
bucket_name = '2025_rides'
fs = gcsfs.GCSFileSystem(project=project_id)

# 3. Integração Mágica: Registra o gcsfs no DuckDB
# Isso faz o DuckDB usar a autenticação do Colab automaticamente
duckdb.register_filesystem(fs)

# 4. Usa o glob do gcsfs para encontrar todos os arquivos parquet na pasta
file_pattern = f"gs://{bucket_name}/outputs_simulation_V6_3/trips_log/_staging/**/*.parquet"
print(f"Buscando arquivos com o padrão: {file_pattern}...")
file_list = fs.glob(file_pattern)
print(f"Encontrados {len(file_list)} arquivos Parquet. Preparando leitura...")

# Adiciona o prefixo gs:// para o DuckDB reconhecer corretamente usando o fsspec/gcsfs
gs_file_list = [f"gs://{f}" for f in file_list]



Buscando arquivos com o padrão: gs://2025_rides/outputs_simulation_V6_3/trips_log/_staging/**/*.parquet...
Encontrados 3531 arquivos Parquet. Preparando leitura...


In [ ]:

# 5. Habilita a barra de progresso do DuckDB
duckdb.sql("PRAGMA enable_progress_bar;")
duckdb.sql("PRAGMA enable_print_progress_bar;")

print("Carregando os dados com DuckDB...")

# 6. Cria a query passando a lista de arquivos e executa retornando para DataFrame pandas
# Limitando a saída de string para evitar erros de formatação na query
files_sql_array = ", ".join([f"'{f}'" for f in gs_file_list])

query = f"""
    SELECT *
    FROM read_parquet([{files_sql_array}])
"""

#df = duckdb.sql(query).df()

#print(f"\nSuccessfully read {len(df):,} rows from GCS parquet files using DuckDB.")
#display(df.head())

Carregando os dados com DuckDB...


In [ ]:
# 1. Autentique sua conta do Google Cloud
#from google.colab import auth
#auth.authenticate_user()

# 2. Defina o ID do seu projeto no GCP e o nome do seu Bucket
#project_id = 'DOUTORADO'
#bucket_name = '2025_rides'

#!gcloud config set project {project_id}

# 3. Monte o seu Google Drive no Colab
#from google.colab import drive
#drive.mount('/content/drive')

#caminhos_parquet = [
#    '/content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/outputs_simulation_V6_3/trips_log/_staging/**/*.parquet',
#    '/content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/outputs_simulation_V6_4/trips_log/_staging/**/*.parquet',
#    '/content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/outputs_simulation_V6_5/trips_log/_staging/**/*.parquet',
#    '/content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/outputs_simulation_V6_6/trips_log/_staging/**/*.parquet',
#    '/content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/outputs_simulation_V6_7/trips_log/_staging/**/*.parquet',
#]

# 4. Copie os dados do Drive direto para o GCS usando o comando otimizado gsutil
# O parâmetro '-m' ativa a cópia em paralelo de múltiplos arquivos
#!gsutil -m cp -r /content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/outputs_simulation_V6_* gs://{bucket_name}/


# INÍCIO DA LEITURA, TRATAMENTO E ANÁLISE DOS DADOS

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
datasets_dir = "/content/drive/MyDrive/DOUTORADO/DATASETS/DATASETS_PRONTOS"
shutil.copy("/content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/arquivos_base/dados_meteorologicos_utci_horario.csv", "./")
shutil.copy("/content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/arquivos_base/DADOS_AEROPORTO/03_voos_atrasados_sbpa.csv", "./")
shutil.copy(f"{datasets_dir}/Aeroporto_Salgado_Filho_h3_res12.csv", "./")

import pandas as pd

# Carregando os datasets
df_clima = pd.read_csv('/content/dados_meteorologicos_utci_horario.csv')
df_voos = pd.read_csv('/content/03_voos_atrasados_sbpa.csv', sep=";")
df_h3 = pd.read_csv('/content/Aeroporto_Salgado_Filho_h3_res12.csv')

# Exibindo as primeiras linhas de cada um para verificar
print("--- Dados Meteorológicos ---")
#display(df_clima.head())

print("\n--- Voos Atrasados ---")
display(df_voos.head())

print("\n--- Aeroporto Salgado Filho (H3) ---")
#display(df_h3.head())

--- Dados Meteorológicos ---

--- Voos Atrasados ---


,ICAO_EMPRESA_AEREA,NUMERO_VOO,CODIGO_AUTORIZACAO_DI,CODIGO_TIPO_LINHA,ICAO_AERODROMO_ORIGEM,ICAO_AERODROMO_DESTINO,PARTIDA_PREVISTA,PARTIDA_REAL,CHEGADA_PREVISTA,CHEGADA_REAL,SITUACAO_VOO,CODIGO_JUSTIFICATIVA,ATRASO_MINUTOS,STATUS_ATRASO,DATA_PREVISTA,DIA_SEMANA,NOME_DIA,HORA_PREVISTA
0,GLO,1885,0,N,SBGR,SBPA,2025-01-23 14:50:00,2025-01-23 15:09:00,2025-01-23 16:35:00,2025-01-23 16:57:00,REALIZADO,NaN,22.0,Atraso Leve (15-59 min),2025-01-23,3,Quinta,16
1,GLO,1885,0,N,SBGR,SBPA,2025-01-26 14:50:00,2025-01-26 15:12:00,2025-01-26 16:35:00,2025-01-26 16:59:00,REALIZADO,NaN,24.0,Atraso Leve (15-59 min),2025-01-26,6,Domingo,16
2,GLO,1885,0,N,SBGR,SBPA,2025-01-29 14:50:00,2025-01-29 14:50:00,2025-01-29 16:35:00,2025-01-29 17:07:00,REALIZADO,NaN,32.0,Atraso Leve (15-59 min),2025-01-29,2,Quarta,16
3,LPE,2422,0,I,SPJC,SBPA,2025-01-06 01:50:00,2025-01-06 02:38:00,2025-01-06 06:30:00,2025-01-06 07:05:00,REALIZADO,NaN,35.0,Atraso Leve (15-59 min),2025-01-06,0,Segunda,6
4,LPE,2422,0,I,SPJC,SBPA,2025-01-20 01:50:00,2025-01-20 02:33:00,2025-01-20 06:30:00,2025-01-20 07:01:00,REALIZADO,NaN,31.0,Atraso Leve (15-59 min),2025-01-20,0,Segunda,6



--- Aeroporto Salgado Filho (H3) ---


In [ ]:
print("--- Tipos de dados: Dados Meteorológicos (df_clima) ---")
#display(df_clima.dtypes)

print("\n--- Tipos de dados: Voos Atrasados (df_voos) ---")
#display(df_voos.dtypes)

print("\n--- Tipos de dados: Aeroporto Salgado Filho H3 (df_h3) ---")
#display(df_h3.dtypes)


--- Tipos de dados: Dados Meteorológicos (df_clima) ---

--- Tipos de dados: Voos Atrasados (df_voos) ---

--- Tipos de dados: Aeroporto Salgado Filho H3 (df_h3) ---


In [ ]:
import gcsfs
import duckdb

# Garante que o filesystem gcsfs esteja registrado no DuckDB
fs = gcsfs.GCSFileSystem(project=project_id)
try:
    duckdb.register_filesystem(fs)
except Exception:
    pass # Ignora caso já esteja registrado

# Caminhos base das simulações
caminhos_base = [
    f'gs://{bucket_name}/outputs_simulation_V6_3/trips_log/_staging/**/*.parquet',
    f'gs://{bucket_name}/outputs_simulation_V6_4/trips_log/_staging/**/*.parquet',
    f'gs://{bucket_name}/outputs_simulation_V6_5/trips_log/_staging/**/*.parquet',
    f'gs://{bucket_name}/outputs_simulation_V6_6/trips_log/_staging/**/*.parquet',
    f'gs://{bucket_name}/outputs_simulation_V6_7/trips_log/_staging/**/*.parquet',
]

print("Buscando todos os arquivos Parquet nos diretórios (isso pode levar alguns instantes)...")
todos_arquivos = []
for caminho in caminhos_base:
    arquivos = fs.glob(caminho)
    todos_arquivos.extend([f"gs://{f}" for f in arquivos])

print(f"Total de {len(todos_arquivos):,} arquivos Parquet encontrados.")

# Criando o array de strings para injetar na query do DuckDB
files_sql_array = ", ".join([f"'{f}'" for f in todos_arquivos])


Buscando todos os arquivos Parquet nos diretórios (isso pode levar alguns instantes)...
Total de 12,368 arquivos Parquet encontrados.


In [ ]:
print("Preparando a query principal para executar com DuckDB...")
print("Lendo do GCS com a nova integração e aplicando undersampling...")

# Habilita a barra de progresso do DuckDB
duckdb.sql("PRAGMA enable_progress_bar;")
duckdb.sql("PRAGMA enable_print_progress_bar;")

# A query balanceada
inicio_ts = 1735699200
fim_ts = 1767235200

# Combine all conditions into a single query with appropriate sampling
# Substituímos {caminhos_parquet} por [{files_sql_array}]
query_combined = f"""
    SELECT
        request_ts,
        event_name,
        origin_h3,
        CASE
            WHEN UPPER(event_name) LIKE '%ATRASADO%' THEN 'DS_VOO'
            WHEN UPPER(event_name) LIKE '%SEVERIDADE%' THEN 'DS_CLIMA'
            ELSE 'DS_OUTROS'
        END AS dataset_type
    FROM read_parquet([{files_sql_array}], hive_partitioning = true)
    WHERE request_ts >= {inicio_ts}
      AND request_ts < {fim_ts}
    USING SAMPLE 30 PERCENT
"""


Preparando a query principal para executar com DuckDB...
Lendo do GCS com a nova integração e aplicando undersampling...


In [ ]:
# Instala a biblioteca necessária
!pip install google-cloud-storage

from google.cloud import storage
from google.colab import auth

# Autenticação
auth.authenticate_user()

# Crie um cliente apontando para o seu projeto
project_id = 'doutorado-501917'
client = storage.Client(project=project_id)

# Acesse o bucket e o arquivo específico
bucket_name = '2025_rides'
bucket = client.get_bucket(bucket_name)

In [ ]:
#!pip install -q gcsfs
#import pandas as pd
#import gcsfs
#from google.colab import auth

# Garante a autenticação nativa do Colab
#auth.authenticate_user()

# Inicializa o FileSystem do GCS apontando para o seu projeto
#fs = gcsfs.GCSFileSystem(project='doutorado-501917')

# Usa o glob do gcsfs para encontrar todos os arquivos parquet na pasta
#file_pattern = f"gs://{bucket_name}/outputs_simulation_V6_3/trips_log/_staging/**/*.parquet"
#print(f"Buscando arquivos com o padrão: {file_pattern}...")

#file_list = fs.glob(file_pattern)
#print(f"Encontrados {len(file_list)} arquivos Parquet. Carregando os dados...")

# Adiciona o prefixo gs:// de volta para o pandas reconhecer e lê a lista
# Passamos o filesystem para evitar problemas de autenticação internos
#df = pd.read_parquet([f"gs://{f}" for f in file_list], filesystem=fs)

#print(f"\nSuccessfully read {len(df):,} rows from GCS parquet files.")
#display(df.head())


In [ ]:
# Executa a query combinada e converte diretamente para DataFrame Pandas

# --- BEGIN FIX: Configure DuckDB for Google Cloud Storage (GCS) ---
# Install and load the httpfs extension for remote file system access
#duckdb.sql("INSTALL httpfs;")
#duckdb.sql("LOAD httpfs;")

# Import necessary libraries for Google Cloud authentication
#import google.auth
#import google.auth.transport.requests

# Get default credentials and refresh them to obtain an access token
#credentials, project = google.auth.default()
#auth_req = google.auth.transport.requests.Request()
#credentials.refresh(auth_req)
#access_token = credentials.token


# 5. Habilita a barra de progresso do DuckDB
duckdb.sql("PRAGMA enable_progress_bar;")
duckdb.sql("PRAGMA enable_print_progress_bar;")

print("Carregando os dados com DuckDB...")

# 6. Cria a query passando a lista de arquivos e executa retornando para DataFrame pandas
# Limitando a saída de string para evitar erros de formatação na query
files_sql_array = ", ".join([f"'{f}'" for f in gs_file_list])

df = duckdb.sql(query_combined).df()

print(f"\nSuccessfully read {len(df):,} rows from GCS parquet files using DuckDB.")
display(df.head())

df_combined = duckdb.sql(query_combined).df()

# Filtra para criar os datasets DS_VOO, DS_CLIMA e DS_OUTROS
DS_VOO = df_combined[df_combined['dataset_type'] == 'DS_VOO'].drop(columns=['dataset_type'])
DS_CLIMA = df_combined[df_combined['dataset_type'] == 'DS_CLIMA'].drop(columns=['dataset_type'])
DS_OUTROS = df_combined[df_combined['dataset_type'] == 'DS_OUTROS'].drop(columns=['dataset_type'])

print(f"🚀 DATASET DE ATRASOS (DS_VOO): {len(DS_VOO):,}")
print(f"🚀 DATASET DE ATRASOS (DS_CLIMA): {len(DS_CLIMA):,}")
print(f"🚀 DATASET DE ATRASOS (DS_OUTROS): {len(DS_OUTROS):,}")
display(DS_VOO.head())

Carregando os dados com DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Successfully read 60,035,671 rows from GCS parquet files using DuckDB.


,request_ts,event_name,origin_h3,dataset_type
0,1735714951,,8ca90139292b1ff,DS_OUTROS
1,1735716014,,8ca9012a48557ff,DS_OUTROS
2,1735718501,,8ca90e9ad0cc3ff,DS_OUTROS
3,1735721351,,8ca90176db4a7ff,DS_OUTROS
4,1735722526,,8ca90128391d3ff,DS_OUTROS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

🚀 DATASET DE ATRASOS (DS_VOO): 199,858
🚀 DATASET DE ATRASOS (DS_CLIMA): 317,707
🚀 DATASET DE ATRASOS (DS_OUTROS): 59,794,468


,request_ts,event_name,origin_h3
3960,1735898854,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e934104bff
4622,1735901421,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e9340b59ff
5938,1735933991,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e935c9a3ff
6236,1735898747,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e935ca81ff
6253,1735900494,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e930d26dff


In [ ]:
import pandas as pd
import numpy as np

print("Iniciando a normalização do tamanho dos datasets DS_VOO, DS_CLIMA e DS_OUTROS...")

# Definir o período de interesse para a distribuição (2025 inteiro)
start_date = pd.to_datetime('2025-01-01 00:00:00')
end_date = pd.to_datetime('2025-12-31 23:59:59')

# Convert UNIX timestamps in seconds to datetime objects
DS_VOO['request_ts_dt'] = pd.to_datetime(DS_VOO['request_ts'], unit='s')
DS_CLIMA['request_ts_dt'] = pd.to_datetime(DS_CLIMA['request_ts'], unit='s')
DS_OUTROS['request_ts_dt'] = pd.to_datetime(DS_OUTROS['request_ts'], unit='s')

print("Corrected datetime format:")
display(DS_VOO[['request_ts', 'request_ts_dt']].head())

Iniciando a normalização do tamanho dos datasets DS_VOO, DS_CLIMA e DS_OUTROS...
Corrected datetime format:


,request_ts,request_ts_dt
3960,1735898854,2025-01-03 10:07:34
4622,1735901421,2025-01-03 10:50:21
5938,1735933991,2025-01-03 19:53:11
6236,1735898747,2025-01-03 10:05:47
6253,1735900494,2025-01-03 10:34:54


In [ ]:
# Filtrar cada DataFrame para o período de 2025
print(f"Filtrando datasets para o período de {start_date.strftime('%Y-%m-%d')} a {end_date.strftime('%Y-%m-%d')}...")
DS_VOO_2025 = DS_VOO[(DS_VOO['request_ts_dt'] >= start_date) & (DS_VOO['request_ts_dt'] <= end_date)]
DS_CLIMA_2025 = DS_CLIMA[(DS_CLIMA['request_ts_dt'] >= start_date) & (DS_CLIMA['request_ts_dt'] <= end_date)]
DS_OUTROS_2025 = DS_OUTROS[(DS_OUTROS['request_ts_dt'] >= start_date) & (DS_OUTROS['request_ts_dt'] <= end_date)]

print(f"Tamanho de DS_VOO_2025 após filtro: {len(DS_VOO_2025):,} linhas")
print(f"Tamanho de DS_CLIMA_2025 após filtro: {len(DS_CLIMA_2025):,} linhas")
print(f"Tamanho de DS_OUTROS_2025 após filtro: {len(DS_OUTROS_2025):,} linhas")

# Encontrar o menor tamanho entre os DataFrames filtrados
min_size = min(len(DS_VOO_2025), len(DS_CLIMA_2025), len(DS_OUTROS_2025))
print(f"\nO menor tamanho entre os datasets filtrados é: {min_size:,} linhas.")

# Realizar o subsampling (amostragem aleatória) para equalizar o tamanho, mantendo a distribuição temporal
print(f"Subsampling todos os datasets para {min_size:,} linhas...")

if len(DS_VOO_2025) > min_size:
    DS_VOO_NORMALIZED = DS_VOO_2025.sample(n=min_size, random_state=42).sort_values('request_ts_dt').reset_index(drop=True)
else:
    DS_VOO_NORMALIZED = DS_VOO_2025.sort_values('request_ts_dt').reset_index(drop=True)

if len(DS_CLIMA_2025) > min_size:
    DS_CLIMA_NORMALIZED = DS_CLIMA_2025.sample(n=min_size, random_state=42).sort_values('request_ts_dt').reset_index(drop=True)
else:
    DS_CLIMA_NORMALIZED = DS_CLIMA_2025.sort_values('request_ts_dt').reset_index(drop=True)

if len(DS_OUTROS_2025) > min_size:
    DS_OUTROS_NORMALIZED = DS_OUTROS_2025.sample(n=min_size, random_state=42).sort_values('request_ts_dt').reset_index(drop=True)
else:
    DS_OUTROS_NORMALIZED = DS_OUTROS_2025.sort_values('request_ts_dt').reset_index(drop=True)

# Atualizar os DataFrames originais com os resultados normalizados
DS_VOO = DS_VOO_NORMALIZED
DS_CLIMA = DS_CLIMA_NORMALIZED
DS_OUTROS = DS_OUTROS_NORMALIZED

print("\n✅ Normalização concluída!")
print(f"Novo tamanho de DS_VOO: {len(DS_VOO):,} linhas")
print(f"Novo tamanho de DS_CLIMA: {len(DS_CLIMA):,} linhas")
print(f"Novo tamanho de DS_OUTROS: {len(DS_OUTROS):,} linhas")

print("\nVerificação das primeiras linhas de cada dataset normalizado:")
print("\nDS_VOO (Normalizado):")
display(DS_VOO.head())

print("\nDS_CLIMA (Normalizado):")
display(DS_CLIMA.head())

print("\nDS_OUTROS (Normalizado):")
display(DS_OUTROS.head())

Filtrando datasets para o período de 2025-01-01 a 2025-12-31...
Tamanho de DS_VOO_2025 após filtro: 199,858 linhas
Tamanho de DS_CLIMA_2025 após filtro: 317,707 linhas
Tamanho de DS_OUTROS_2025 após filtro: 59,794,468 linhas

O menor tamanho entre os datasets filtrados é: 199,858 linhas.
Subsampling todos os datasets para 199,858 linhas...

✅ Normalização concluída!
Novo tamanho de DS_VOO: 199,858 linhas
Novo tamanho de DS_CLIMA: 199,858 linhas
Novo tamanho de DS_OUTROS: 199,858 linhas

Verificação das primeiras linhas de cada dataset normalizado:

DS_VOO (Normalizado):


,request_ts,event_name,origin_h3,request_ts_dt
0,1735898423,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e9358d99ff,2025-01-03 10:00:23
1,1735898430,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e9341731ff,2025-01-03 10:00:30
2,1735898470,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e934b651ff,2025-01-03 10:01:10
3,1735898471,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90129b4dadff,2025-01-03 10:01:11
4,1735898477,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e9358c3dff,2025-01-03 10:01:17



DS_CLIMA (Normalizado):


,request_ts,event_name,origin_h3,request_ts_dt
0,1735718408,Aeroporto Salgado Filho Weather Severidade 5,8ca90e935ce53ff,2025-01-01 08:00:08
1,1735718413,Aeroporto Salgado Filho Weather Severidade 5,8ca90e9342415ff,2025-01-01 08:00:13
2,1735718437,Aeroporto Salgado Filho Weather Severidade 5,8ca90e935c065ff,2025-01-01 08:00:37
3,1735718452,Aeroporto Salgado Filho Weather Severidade 5,8ca90e9358ccdff,2025-01-01 08:00:52
4,1735718515,Aeroporto Salgado Filho Weather Severidade 5,8ca90e9358e51ff,2025-01-01 08:01:55



DS_OUTROS (Normalizado):


,request_ts,event_name,origin_h3,request_ts_dt
0,1735704844,,8ca901298a137ff,2025-01-01 04:14:04
1,1735705452,,8ca90e91b235bff,2025-01-01 04:24:12
2,1735705753,,8ca90e8021765ff,2025-01-01 04:29:13
3,1735705838,,8ca90e835946bff,2025-01-01 04:30:38
4,1735705994,,8ca90e9030147ff,2025-01-01 04:33:14


In [ ]:
print(f"🚀 DATASET DE EVENTOS CLIMÁTICOS (DS_CLIMA): {len(DS_CLIMA):,}")
display(DS_CLIMA.head())

🚀 DATASET DE EVENTOS CLIMÁTICOS (DS_CLIMA): 199,858


,request_ts,event_name,origin_h3,request_ts_dt
0,1735718408,Aeroporto Salgado Filho Weather Severidade 5,8ca90e935ce53ff,2025-01-01 08:00:08
1,1735718413,Aeroporto Salgado Filho Weather Severidade 5,8ca90e9342415ff,2025-01-01 08:00:13
2,1735718437,Aeroporto Salgado Filho Weather Severidade 5,8ca90e935c065ff,2025-01-01 08:00:37
3,1735718452,Aeroporto Salgado Filho Weather Severidade 5,8ca90e9358ccdff,2025-01-01 08:00:52
4,1735718515,Aeroporto Salgado Filho Weather Severidade 5,8ca90e9358e51ff,2025-01-01 08:01:55


In [ ]:
# Executa a query e converte diretamente para DataFrame Pandas
print(f"🚀 DATASET DE OUTROS EVENTOS (DS_OUTROS): {len(DS_OUTROS):,}")
display(DS_OUTROS.head())

🚀 DATASET DE OUTROS EVENTOS (DS_OUTROS): 199,858


,request_ts,event_name,origin_h3,request_ts_dt
0,1735704844,,8ca901298a137ff,2025-01-01 04:14:04
1,1735705452,,8ca90e91b235bff,2025-01-01 04:24:12
2,1735705753,,8ca90e8021765ff,2025-01-01 04:29:13
3,1735705838,,8ca90e835946bff,2025-01-01 04:30:38
4,1735705994,,8ca90e9030147ff,2025-01-01 04:33:14


In [ ]:
len_atraso = len(DS_VOO)
len_climatico = len(DS_CLIMA)
len_nao_evento = len(DS_OUTROS)

total = len_atraso + len_climatico + len_nao_evento

print(f"Total de linhas: {total:,}\n")
print(f"Atraso de Voo (DS_VOO): {len_atraso:,} ({len_atraso/total:.2%})")
print(f"Evento Climático (DS_CLIMA): {len_climatico:,} ({len_climatico/total:.2%})")
print(f"Outros Eventos (DS_OUTROS): {len_nao_evento:,} ({len_nao_evento/total:.2%})")

Total de linhas: 599,574

Atraso de Voo (DS_VOO): 199,858 (33.33%)
Evento Climático (DS_CLIMA): 199,858 (33.33%)
Outros Eventos (DS_OUTROS): 199,858 (33.33%)


In [ ]:
# Convertendo request_ts para datetime nos 3 datasets
#DS_VOO['request_ts_dt'] = pd.to_datetime(DS_VOO['request_ts'], unit='s')
#DS_CLIMA['request_ts_dt'] = pd.to_datetime(DS_CLIMA['request_ts'], unit='s')
#DS_OUTROS['request_ts_dt'] = pd.to_datetime(DS_OUTROS['request_ts'], unit='s')

# Exibindo uma amostra rápida de cada um para verificação
print("Atrasos de Voo (DS_VOO):")
display(DS_VOO[['request_ts', 'request_ts_dt']].head(2))

print("\nEventos Climáticos (DS_CLIMA):")
display(DS_CLIMA[['request_ts', 'request_ts_dt']].head(2))

print("\nOutros Eventos (DS_OUTROS):")
display(DS_OUTROS[['request_ts', 'request_ts_dt']].head(2))

Atrasos de Voo (DS_VOO):


,request_ts,request_ts_dt
0,1735898423,2025-01-03 10:00:23
1,1735898430,2025-01-03 10:00:30



Eventos Climáticos (DS_CLIMA):


,request_ts,request_ts_dt
0,1735718408,2025-01-01 08:00:08
1,1735718413,2025-01-01 08:00:13



Outros Eventos (DS_OUTROS):


,request_ts,request_ts_dt
0,1735704844,2025-01-01 04:14:04
1,1735705452,2025-01-01 04:24:12


### 🚀 Estratégia para Criação do Algoritmo Preditivo

Com base nas análises feitas (Clima + Voos), a melhor estratégia para sair de modelos *explicativos* para um modelo **preditivo** robusto envolve 5 pilares fundamentais:

#### 1. Criação da "Base Master" (Unificação)
Até agora, avaliamos o clima e os voos separadamente. O modelo preditivo precisará de tudo na mesma linha do tempo.
*   **Ação:** Fazer um merge combinando o `df_ts` (Eventos) com o `df_clima` (variáveis meteorológicas topo) **E** agregando informações do `df_voos` (ex: número de voos previstos para pousar naquela janela de 2 horas, mix de empresas aéreas, etc.).

#### 2. Engenharia de Features (Feature Engineering)
Além dos dados brutos, precisamos dar contexto ao algoritmo.
*   **Temporais:** Extrair `Hora do Dia`, `Mês`, `Dia da Semana`, e `Trimestre` (como vimos, o Q3 é crítico).
*   **Lags (Defasagens):** Usar o clima das horas *anteriores* para prever a próxima hora (ex: se a pressão atmosférica está caindo nas últimas 3 horas, a chance de evento aumenta).

#### 3. Tratamento de Desbalanceamento Severo
O seu *target* (eventos) representa menos de 1% da base total (ex: 38k contra 4.4M). Se não tratarmos, o modelo vai sempre prever "0" e acertar 99% das vezes, mas falhar no que importa.
*   **Ação:** Usar parâmetros nativos como `is_unbalance=True` ou `scale_pos_weight` no LightGBM/XGBoost.
*   **Alternativa:** Técnicas de reamostragem como *Undersampling* da classe majoritária no treino (você já fez um sampling para evitar OOM, podemos otimizar isso).

#### 4. Validação Temporal (Time-Series Split)
**Nunca** use um `train_test_split` aleatório em dados que dependem do tempo, pois isso gera *Data Leakage* (vazar o futuro para prever o passado).
*   **Ação:** Separar os dados cronologicamente. Treinar com os primeiros 9 meses de 2025 (Jan-Set) e testar com os 3 meses seguintes (Out-Dez) para simular o uso no mundo real.

#### 5. Métricas de Avaliação Corretas
*Acurácia* não serve aqui.
*   Focar no **Recall** (capacidade de detectar todos os eventos reais, minimizando falsos negativos) e **Precision** (quando dá o alerta, qual a chance de ser real).
*   Analisar a curva **PR-AUC** (Precision-Recall Area Under Curve), que é a métrica padrão-ouro para dados altamente desbalanceados.

---
**Próximo Passo Prático:** gerar código para construir a **Base Master** unindo Clima + Voos + features temporais?

In [ ]:
import pandas as pd
import numpy as np

print("1. Padronizando as colunas temporais e criando janelas de 4 horas...")
# Garantir que as colunas de tempo sejam datetime
df_clima['time'] = pd.to_datetime(df_clima['time'])
df_voos['CHEGADA_REAL'] = pd.to_datetime(df_voos['CHEGADA_REAL'], errors='coerce')

# Criar as janelas de 4 horas
df_clima['time_window'] = df_clima['time'].dt.floor('4h')
df_voos['time_window'] = df_voos['CHEGADA_REAL'].dt.floor('4h')

# Garantir datetime e criar janelas para os eventos
dataframes_eventos = [DS_OUTROS, DS_CLIMA, DS_VOO]
for df in dataframes_eventos:
    df['request_ts_dt'] = pd.to_datetime(df['request_ts_dt'])
    df['time_window'] = df['request_ts_dt'].dt.floor('4h')

print("2. Definindo o TARGET numérico para cada classe...")
# 0: Não Evento | 1: Evento Climático | 2: Atraso de Voo
DS_OUTROS['target'] = 0
DS_CLIMA['target'] = 1
DS_VOO['target'] = 2

# Concatenar todos os eventos na base principal
df_eventos_combinados = pd.concat([
    DS_OUTROS[['time_window', 'origin_h3', 'target', 'request_ts_dt']],
    DS_CLIMA[['time_window', 'origin_h3', 'target', 'request_ts_dt']],
    DS_VOO[['time_window', 'origin_h3', 'target', 'request_ts_dt']]
], ignore_index=True)

print("3. Agregando Clima e Voos nas janelas...")
# Clima: tira a média das variáveis meteorológicas nas janelas
df_clima_agg = df_clima.drop(columns=['time']).groupby('time_window').mean(numeric_only=True).reset_index()

# Voos: contabiliza e preserva as informações detalhadas em listas para cada janela
df_voos_agg = df_voos.dropna(subset=['time_window']).groupby('time_window').agg(
    qtd_voos_previstos=('NUMERO_VOO', 'count'),
    qtd_empresas_aereas=('ICAO_EMPRESA_AEREA', lambda x: x.nunique()),
    lista_chegada_real=('CHEGADA_REAL', lambda x: list(x)),
    lista_empresas_aereas=('ICAO_EMPRESA_AEREA', lambda x: list(x)),
    lista_numeros_voo=('NUMERO_VOO', lambda x: list(x)),
    lista_codigo_linha=('CODIGO_TIPO_LINHA', lambda x: list(x))
).reset_index()

print("4. Montando a Base Master (Merges)...")
# Unindo Eventos + Clima Agregado
df_master = pd.merge(df_eventos_combinados, df_clima_agg, on='time_window', how='left')

# Unindo com os Voos Agregados
df_master = pd.merge(df_master, df_voos_agg, on='time_window', how='left')

# Preenchendo nulos onde não havia voos na janela com 0
df_master['qtd_voos_previstos'] = df_master['qtd_voos_previstos'].fillna(0)
df_master['qtd_empresas_aereas'] = df_master['qtd_empresas_aereas'].fillna(0)

# Ordenar a base cronologicamente
df_master = df_master.sort_values('time_window').reset_index(drop=True)

print("\n✅ Base Master Finalizada e pronta para ML!")
print(f"Tamanho final da base: {len(df_master):,} linhas e {df_master.shape[1]} colunas")
print("\nDistribuição do Target:")
target_map = {0: '0 (Não Evento)', 1: '1 (Evento Climático)', 2: '2 (Atraso de Voo)'}
print(df_master['target'].map(target_map).value_counts())

display(df_master.head())


1. Padronizando as colunas temporais e criando janelas de 4 horas...
2. Definindo o TARGET numérico para cada classe...
3. Agregando Clima e Voos nas janelas...
4. Montando a Base Master (Merges)...

✅ Base Master Finalizada e pronta para ML!
Tamanho final da base: 599,574 linhas e 54 colunas

Distribuição do Target:
target
0 (Não Evento)          199858
1 (Evento Climático)    199858
2 (Atraso de Voo)       199858
Name: count, dtype: int64


,time_window,origin_h3,target,request_ts_dt,temperature_2m,relative_humidity_2m,wind_speed_10m,shortwave_radiation,direct_radiation,diffuse_radiation,...,utci_has_heat_stress,utci_has_cold_stress,utci_has_strong_heat_stress,utci_has_strong_cold_stress,qtd_voos_previstos,qtd_empresas_aereas,lista_chegada_real,lista_empresas_aereas,lista_numeros_voo,lista_codigo_linha
0,2025-01-01 04:00:00,8ca901298a137ff,0,2025-01-01 04:14:04,20.586111,96.694444,1.885833,33.833333,15.638889,18.194444,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
1,2025-01-01 04:00:00,8ca90e91c9669ff,0,2025-01-01 06:50:44,20.586111,96.694444,1.885833,33.833333,15.638889,18.194444,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
2,2025-01-01 04:00:00,8ca9012998f33ff,0,2025-01-01 06:50:58,20.586111,96.694444,1.885833,33.833333,15.638889,18.194444,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
3,2025-01-01 04:00:00,8ca90128a406dff,0,2025-01-01 06:53:45,20.586111,96.694444,1.885833,33.833333,15.638889,18.194444,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
4,2025-01-01 04:00:00,8ca90e82679bbff,0,2025-01-01 06:53:46,20.586111,96.694444,1.885833,33.833333,15.638889,18.194444,...,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN


In [ ]:
for coluna in df_master.columns:
    print(coluna)

time_window
origin_h3
target
request_ts_dt
temperature_2m
relative_humidity_2m
wind_speed_10m
shortwave_radiation
direct_radiation
diffuse_radiation
direct_normal_irradiance
sunshine_duration
cloud_cover
dew_point_2m
apparent_temperature
precipitation
rain
weather_code
cloud_cover_low
cloud_cover_mid
cloud_cover_high
wind_direction_10m
wind_gusts_10m
surface_pressure
pressure_msl
vapour_pressure_deficit
api_latitude
api_longitude
api_elevation
api_utc_offset_seconds
LAT
LONG
ELEVATION
hour
utci_tdb_c
utci_rh_pct
utci_wind_speed_10m_mps_raw
utci_is_day_estimated
utci_radiative_adjustment_c
utci_tr_c
utci_wind_speed_10m_mps_used
utci_wind_was_clipped
utci_c
utci_discomfort_score_0_100
utci_has_heat_stress
utci_has_cold_stress
utci_has_strong_heat_stress
utci_has_strong_cold_stress
qtd_voos_previstos
qtd_empresas_aereas
lista_chegada_real
lista_empresas_aereas
lista_numeros_voo
lista_codigo_linha


In [ ]:
import pandas as pd

print("1. Criando Features Temporais...")
df_master['hora'] = df_master['time_window'].dt.hour
df_master['mes'] = df_master['time_window'].dt.month
df_master['dia_semana'] = df_master['time_window'].dt.dayofweek
df_master['trimestre'] = df_master['time_window'].dt.quarter

print("2. Criando Features de Lag (Defasagem) no Clima...")
# Ordenar a base agregada de clima por tempo para garantir o shift temporal correto
df_clima_agg = df_clima_agg.sort_values('time_window')

# Selecionar variáveis climáticas chaves para analisar a mudança nas últimas 4 horas
cols_clima_para_lag = ['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m']
if 'surface_pressure' in df_clima_agg.columns:
    cols_clima_para_lag.append('surface_pressure')

# Calcular o lag de 1 período (como as janelas são de 4h, lag 1 = 4 horas antes)
for col in cols_clima_para_lag:
    df_clima_agg[f'{col}_lag4h'] = df_clima_agg[col].shift(1)

print("3. Integrando as novas variáveis de lag ao df_master...")
colunas_lags = ['time_window'] + [f'{col}_lag4h' for col in cols_clima_para_lag]

# Realizar o merge usando a mesma janela de tempo
df_master = pd.merge(df_master, df_clima_agg[colunas_lags], on='time_window', how='left')

print("\n--- Amostra das Novas Features ---")
cols_amostra = ['time_window', 'hora', 'mes', 'dia_semana', 'trimestre'] + [f'{col}_lag4h' for col in cols_clima_para_lag]
display(df_master[cols_amostra].head())

1. Criando Features Temporais...
2. Criando Features de Lag (Defasagem) no Clima...
3. Integrando as novas variáveis de lag ao df_master...

--- Amostra das Novas Features ---


,time_window,hora,mes,dia_semana,trimestre,temperature_2m_lag4h,relative_humidity_2m_lag4h,wind_speed_10m_lag4h,surface_pressure_lag4h
0,2025-01-01 04:00:00,4,1,2,1,20.919444,95.472222,2.75,1002.383333
1,2025-01-01 04:00:00,4,1,2,1,20.919444,95.472222,2.75,1002.383333
2,2025-01-01 04:00:00,4,1,2,1,20.919444,95.472222,2.75,1002.383333
3,2025-01-01 04:00:00,4,1,2,1,20.919444,95.472222,2.75,1002.383333
4,2025-01-01 04:00:00,4,1,2,1,20.919444,95.472222,2.75,1002.383333


# 🧠 Pipeline de Explainable AI (XAI) — LightGBM + SHAP

A partir daqui construímos o **melhor modelo interpretável possível** sobre a `df_master`, seguindo boas práticas de ML e, sobretudo, de **IA Explicável (XAI)**.

**Arquitetura do pipeline:**

1. **Seleção de features** — remove identificadores, colunas de listas (dtype `object`), constantes (aeroporto único) e intermediários redundantes do UTCI.
2. **Validação temporal** — split cronológico (sem *data leakage*), com sub-split interno para *early stopping*.
3. **Desbalanceamento** — pesos de classe reforçados para priorizar a detecção de eventos (classes 1 e 2).
4. **Baseline → Optuna → Modelo final** — otimização de hiperparâmetros maximizando *macro-F1*.
5. **Avaliação** — `classification_report`, PR-AUC (One-vs-Rest) e matriz de confusão.
6. **🔍 SHAP (núcleo XAI)** — explicações **globais** (importância por classe, *beeswarm*, *dependence*) e **locais** (*waterfall* de casos individuais).
7. **Calibração + Persistência** — *reliability diagram* e salvamento do modelo para produção.

> **Por que SHAP?** É o padrão-ouro em XAI: fundamentado em valores de Shapley (teoria dos jogos), decompõe cada previsão na contribuição aditiva de cada variável, permitindo explicar tanto o comportamento **global** do modelo quanto **cada decisão individual**.

In [ ]:
# ============================================================
# 1. SELEÇÃO DE FEATURES E PREPARAÇÃO DOS DADOS
# ============================================================
!pip install -q shap optuna

import numpy as np
import pandas as pd
import lightgbm as lgb
from lightgbm import LGBMClassifier
import shap
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
nomes_classes = ['0 (Não Evento)', '1 (Evento Climático)', '2 (Atraso de Voo)']

# Ordenação cronológica é pré-requisito para a validação temporal
df_master = df_master.sort_values('time_window').reset_index(drop=True)

# Colunas que NÃO são features preditivas:
#  - identificadores / target / tempo
#  - listas (dtype object → incompatível com LightGBM)
#  - constantes (aeroporto único → variância zero): lat/long/elevação/offset
#  - 'hour' (duplicata de 'hora' derivada da janela)
#  - intermediários do cálculo UTCI (redundantes com utci_c / utci_discomfort_score)
cols_bloqueadas = {
    'time_window', 'origin_h3', 'target', 'request_ts_dt',
    'lista_chegada_real', 'lista_empresas_aereas', 'lista_numeros_voo', 'lista_codigo_linha',
    'api_latitude', 'api_longitude', 'api_elevation', 'api_utc_offset_seconds',
    'LAT', 'LONG', 'ELEVATION', 'hour',
    'utci_tdb_c', 'utci_rh_pct', 'utci_wind_speed_10m_mps_raw', 'utci_is_day_estimated',
    'utci_radiative_adjustment_c', 'utci_tr_c', 'utci_wind_speed_10m_mps_used', 'utci_wind_was_clipped',
}

# Mantém apenas colunas numéricas e não bloqueadas
FEATURES = [
    c for c in df_master.columns
    if c not in cols_bloqueadas and pd.api.types.is_numeric_dtype(df_master[c])
]

# Descarta colunas de variância zero (não informam o modelo e poluem o SHAP)
variancias = df_master[FEATURES].var(numeric_only=True)
FEATURES = [c for c in FEATURES if variancias.get(c, 0) > 0]

# Preenche NaNs dos lags (primeiras janelas) de forma temporalmente coerente
df_master[FEATURES] = df_master[FEATURES].ffill().bfill()

X = df_master[FEATURES].copy()
y = df_master['target'].copy()

print(f"✅ {len(FEATURES)} features selecionadas para o modelo XAI:\n")
for f in FEATURES:
    print(f"   • {f}")
print(f"\nShape de X: {X.shape}")
print(f"\nDistribuição do target:\n{y.map(target_map).value_counts()}")

In [ ]:
# ============================================================
# 2. VALIDAÇÃO TEMPORAL — Split cronológico (sem data leakage)
# ============================================================
indice_corte = int(len(df_master) * 0.80)

df_train = df_master.iloc[:indice_corte]
df_test  = df_master.iloc[indice_corte:]

X_train, y_train = X.iloc[:indice_corte], y.iloc[:indice_corte]
X_test,  y_test  = X.iloc[indice_corte:], y.iloc[indice_corte:]

# Sub-split cronológico interno ao treino para early stopping / Optuna
# (nunca usamos o conjunto de teste para tomar decisões de treino)
corte_val = int(len(X_train) * 0.85)
X_tr,  y_tr  = X_train.iloc[:corte_val], y_train.iloc[:corte_val]
X_val, y_val = X_train.iloc[corte_val:], y_train.iloc[corte_val:]

print("Períodos (validação temporal):")
print(f"  Treino : {df_train['time_window'].min()}  →  {df_train['time_window'].max()}  ({len(X_train):,} amostras)")
print(f"  Teste  : {df_test['time_window'].min()}  →  {df_test['time_window'].max()}  ({len(X_test):,} amostras)")
print(f"  Val interno (early stopping): {len(X_val):,} amostras")

In [ ]:
# ============================================================
# 3. TRATAMENTO DE DESBALANCEAMENTO — Pesos de classe
# ============================================================
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import label_binarize

classes_unicas = np.unique(y_train)
pesos = compute_class_weight(class_weight='balanced', classes=classes_unicas, y=y_train)
class_weight_dict = dict(zip(classes_unicas, pesos))

# Reforço manual: prioriza a detecção dos eventos (classes 1 e 2) sobre 'Não Evento'
class_weight_dict[0] *= 0.5
class_weight_dict[1] *= 1.5
class_weight_dict[2] *= 1.5

# Pesos por amostra para o treino (usado no sub-split X_tr)
sample_weights_tr = y_tr.map(class_weight_dict)

# Target binarizado (One-vs-Rest) para PR-AUC
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])

print("Pesos aplicados por classe:")
for c, w in class_weight_dict.items():
    print(f"   {target_map[c]}: {w:.3f}")

## 4. Modelo Baseline

Um LightGBM com parâmetros padrão estabelece o **piso de desempenho**. Tudo o que vier depois (Optuna, features) precisa superá-lo para justificar a complexidade adicional.

In [ ]:
from sklearn.metrics import classification_report, average_precision_score, f1_score

model_base = LGBMClassifier(
    objective='multiclass', num_class=3, n_estimators=400,
    learning_rate=0.05, random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1,
)
model_base.fit(
    X_tr, y_tr, sample_weight=sample_weights_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(30, verbose=False)],
)

proba_base = model_base.predict_proba(X_test)
pred_base = np.argmax(proba_base, axis=1)

print("="*55)
print("📊 BASELINE — Classification Report (conjunto de teste)")
print("="*55)
print(classification_report(y_test, pred_base, target_names=nomes_classes, zero_division=0))
print(f"Macro-F1 baseline: {f1_score(y_test, pred_base, average='macro'):.4f}")

## 5. Otimização de Hiperparâmetros com Optuna

Busca bayesiana maximizando o **macro-F1** no conjunto de validação interno (`X_val`). O macro-F1 dá peso igual às três classes, evitando que o modelo se especialize apenas na classe mais fácil.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'objective': 'multiclass', 'num_class': 3, 'verbosity': -1,
        'n_estimators': 600, 'random_state': RANDOM_STATE, 'n_jobs': -1,
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves':       trial.suggest_int('num_leaves', 20, 128),
        'max_depth':        trial.suggest_int('max_depth', 3, 12),
        'min_child_samples':trial.suggest_int('min_child_samples', 10, 120),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq':     trial.suggest_int('bagging_freq', 1, 7),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }
    m = LGBMClassifier(**params)
    m.fit(
        X_tr, y_tr, sample_weight=sample_weights_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(30, verbose=False)],
    )
    pred = np.argmax(m.predict_proba(X_val), axis=1)
    return f1_score(y_val, pred, average='macro')

print("Iniciando a busca de hiperparâmetros (pode levar vários minutos)...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=25, show_progress_bar=True)

best_params = study.best_params.copy()
best_params.update({
    'objective': 'multiclass', 'num_class': 3, 'random_state': RANDOM_STATE,
    'n_jobs': -1, 'verbosity': -1, 'n_estimators': 800,
})

print(f"\n✅ Melhor macro-F1 (val): {study.best_value:.4f}")
print("Melhores hiperparâmetros:")
for k, v in best_params.items():
    print(f"   {k}: {v}")

In [ ]:
# ============================================================
# 6. MODELO FINAL — treino com os melhores hiperparâmetros
# ============================================================
final_model = LGBMClassifier(**best_params)
final_model.fit(
    X_tr, y_tr, sample_weight=sample_weights_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(40, verbose=False)],
)
print(f"✅ Modelo final treinado. Melhor iteração: {final_model.best_iteration_}")

# Previsões no conjunto de teste (holdout temporal)
y_pred_prob_final = final_model.predict_proba(X_test)
y_pred_final = np.argmax(y_pred_prob_final, axis=1)

## 7. Avaliação do Modelo Final

Como o *target* é balanceado 1:1:1, reportamos `classification_report` (precision/recall/F1 por classe), **PR-AUC** One-vs-Rest (robusta a desbalanceamento) e a **matriz de confusão**. Foco especial no **recall da classe 2 (Atraso de Voo)** — capacidade de não deixar passar atrasos reais.

In [ ]:
from sklearn.metrics import confusion_matrix

print("="*55)
print("📊 MODELO FINAL — Classification Report (conjunto de teste)")
print("="*55)
print(classification_report(y_test, y_pred_final, target_names=nomes_classes, zero_division=0))
print(f"Macro-F1 final: {f1_score(y_test, y_pred_final, average='macro'):.4f}")

print("\n--- PR-AUC (Área sob a curva Precision-Recall, One-vs-Rest) ---")
for i, cn in enumerate(nomes_classes):
    pr_auc = average_precision_score(y_test_bin[:, i], y_pred_prob_final[:, i])
    print(f"   {cn}: {pr_auc:.4f}")

# Matriz de confusão
cm = confusion_matrix(y_test, y_pred_final)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=nomes_classes, yticklabels=nomes_classes)
plt.title('Matriz de Confusão — Modelo Final', fontsize=13)
plt.ylabel('Classe Real'); plt.xlabel('Classe Prevista')
plt.tight_layout(); plt.show()

# Detalhamento dos erros na classe crítica (Atraso de Voo)
fp_atraso = cm[0, 2] + cm[1, 2]   # previu atraso, mas não era
fn_atraso = cm[2, 0] + cm[2, 1]   # era atraso, mas não previu
print(f"\n🔎 Classe 2 (Atraso de Voo):")
print(f"   Falsos Positivos (alarme falso): {fp_atraso:,}")
print(f"   Falsos Negativos (atraso perdido): {fn_atraso:,}")

## 8. 🔍 Explicabilidade com SHAP — o núcleo da XAI

Aqui está o coração da IA Explicável. Usamos o **`TreeExplainer`** (exato e rápido para modelos de árvore) para decompor as previsões em contribuições aditivas de cada variável (valores de Shapley).

Produzimos três níveis de explicação:

- **Global (importância por classe):** quais variáveis mais pesam na decisão de cada classe.
- **Global (*beeswarm*):** como o valor de cada variável (alto/baixo) empurra a previsão.
- **Local (*waterfall*):** por que o modelo tomou **uma** decisão específica.

In [ ]:
# ============================================================
# 8.1 Cálculo dos valores SHAP (TreeExplainer)
# ============================================================
# Amostra do teste para acelerar (SHAP é O(nº árvores × nós))
n_shap = min(5000, len(X_test))
X_shap = X_test.sample(n=n_shap, random_state=RANDOM_STATE).reset_index(drop=True)

explainer = shap.TreeExplainer(final_model)
shap_raw = explainer.shap_values(X_shap)

# Normaliza para lista de arrays 2D (um por classe) — robusto entre versões do SHAP
if isinstance(shap_raw, list):
    shap_por_classe = shap_raw                                  # SHAP antigo: lista por classe
else:
    shap_por_classe = [shap_raw[:, :, i] for i in range(shap_raw.shape[2])]  # SHAP novo: array 3D

# Valor esperado (base) por classe
expected = explainer.expected_value
expected = np.atleast_1d(expected)
if len(expected) == 1:
    expected = np.repeat(expected, len(shap_por_classe))

print(f"✅ SHAP calculado: {n_shap:,} amostras × {len(FEATURES)} features × {len(shap_por_classe)} classes.")

In [ ]:
# ============================================================
# 8.2 Importância Global SHAP — média |SHAP| por classe
# ============================================================
mean_abs = np.array([np.abs(sv).mean(axis=0) for sv in shap_por_classe])  # (classes, features)
df_imp = pd.DataFrame(mean_abs.T, index=FEATURES, columns=nomes_classes)
df_imp['Total'] = df_imp.sum(axis=1)
df_imp = df_imp.sort_values('Total', ascending=False)

top20 = df_imp.head(20)
top20[nomes_classes].iloc[::-1].plot(
    kind='barh', stacked=True, figsize=(10, 8), colormap='viridis'
)
plt.title('Importância Global SHAP — Top 20 (média |SHAP| por classe)', fontsize=13)
plt.xlabel('Impacto médio na saída do modelo (|SHAP|)')
plt.ylabel('Variável')
plt.legend(title='Classe', loc='lower right')
plt.tight_layout(); plt.show()

print("Top 10 variáveis mais influentes (global):")
display(df_imp.head(10).round(4))

In [ ]:
# ============================================================
# 8.3 Beeswarm SHAP — distribuição do impacto por classe
# ============================================================
# Cada ponto é uma amostra; a cor indica o valor da feature (vermelho=alto, azul=baixo).
for i, cn in enumerate(nomes_classes):
    print(f"\n{'='*55}\nSHAP Beeswarm — {cn}\n{'='*55}")
    shap.summary_plot(
        shap_por_classe[i], X_shap, feature_names=FEATURES,
        max_display=15, show=False,
    )
    plt.title(f'SHAP — Impacto das variáveis na classe: {cn}', fontsize=12)
    plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# 8.4 Dependence Plots — efeito das top features na classe ATRASO DE VOO
# ============================================================
# Mostra como o valor de uma variável se relaciona com seu impacto SHAP,
# revelando relações não-lineares e interações (cor = feature que mais interage).
classe_alvo = 2  # Atraso de Voo
top_feats = df_imp.index[:3].tolist()

for feat in top_feats:
    shap.dependence_plot(
        feat, shap_por_classe[classe_alvo], X_shap,
        feature_names=FEATURES, show=False,
    )
    plt.title(f'Dependence — "{feat}" → impacto em {nomes_classes[classe_alvo]}', fontsize=12)
    plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# 8.5 Explicação LOCAL — por que o modelo decidiu por um caso específico
# ============================================================
# Seleciona um exemplo que o modelo previu como Atraso de Voo (classe 2)
pred_shap = np.argmax(final_model.predict_proba(X_shap), axis=1)
idxs = np.where(pred_shap == 2)[0]
idx = int(idxs[0]) if len(idxs) else 0
classe = int(pred_shap[idx])

expl = shap.Explanation(
    values=shap_por_classe[classe][idx],
    base_values=expected[classe],
    data=X_shap.iloc[idx].values,
    feature_names=FEATURES,
)

print(f"Explicação local — amostra #{idx} | classe prevista: {nomes_classes[classe]}")
print("Contribuições que empurram para (+) ou contra (−) essa classe:\n")
shap.plots.waterfall(expl, max_display=15, show=True)

## 9. Calibração de Probabilidades e Persistência

O *reliability diagram* verifica se as probabilidades do modelo são confiáveis (uma previsão de "80%" deve acertar ~80% das vezes). Em seguida salvamos o modelo, a lista de features e demonstramos a **inferência em produção**.

In [ ]:
from sklearn.calibration import calibration_curve

plt.figure(figsize=(8, 6))
for i, cn in enumerate(nomes_classes):
    prob_true, prob_pred = calibration_curve(
        (y_test == i).astype(int), y_pred_prob_final[:, i], n_bins=10
    )
    plt.plot(prob_pred, prob_true, marker='o', label=cn)

plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Calibração perfeita')
plt.title('Curva de Calibração (Reliability Diagram)', fontsize=13)
plt.xlabel('Probabilidade prevista média')
plt.ylabel('Fração real de positivos')
plt.legend(); plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# 9.1 Persistência do modelo + inferência explicável em produção
# ============================================================
import joblib

final_model.booster_.save_model('modelo_xai_lgb_v1.txt')   # formato nativo LightGBM
joblib.dump(final_model, 'modelo_xai_lgb_v1.pkl')           # estimador sklearn completo
joblib.dump(FEATURES, 'features_xai_v1.pkl')                # ordem exata das features
print("✅ Artefatos salvos: modelo_xai_lgb_v1.{txt,pkl} + features_xai_v1.pkl")

# --- Simulação de inferência para uma nova janela de 4h ---
exemplo = X_test.iloc[[10]][FEATURES]
proba = final_model.predict_proba(exemplo)[0]
classe = int(np.argmax(proba))

print("\n🔮 PREVISÃO PARA A JANELA:")
for i, cn in enumerate(nomes_classes):
    print(f"   {cn}: {proba[i]:.1%}")
print("-" * 45)
print(f"🚨 DECISÃO DO MODELO: {target_map[classe]}")

# Explicação SHAP local da decisão (transparência em produção)
sv_ex = explainer.shap_values(exemplo)
sv_ex = sv_ex if isinstance(sv_ex, list) else [sv_ex[:, :, k] for k in range(sv_ex.shape[2])]
contrib = pd.Series(sv_ex[classe][0], index=FEATURES).sort_values(key=np.abs, ascending=False)
print(f"\n📋 Top 5 fatores que levaram a esta decisão ({nomes_classes[classe]}):")
for feat, val in contrib.head(5).items():
    direcao = "↑ favorece" if val > 0 else "↓ contra"
    print(f"   {direcao}  {feat}: SHAP={val:+.4f}  (valor={exemplo.iloc[0][feat]:.3f})")